## Load Data
**Only run as needed**

In [57]:
import pandas as pd
import regex as re
import json


## Data Cleaning
**Only run as needed**

In [63]:
courses = pd.read_csv('courses2.csv')
# rename columns
courses = courses.rename(columns={
    'Catalog #': 'catalog_number',
    'Course Title Long \n(Short)': 'course_title_long_short',
    'Current Enforced Prerequisites': 'prerequisites',
    'Current Advisory Prerequisites': 'advisory_prerequisites',
    'Course Description': 'course_description',
    "Term Offerings \n2025-26AY": 'term_offerings_2025_26',
    'Term Offerings \n2022-23AY': 'term_offerings_2022_23',
    'Typical Term Offered': 'typical_term_offered',
    'Last Term Offered': 'last_term_offered',
    'Cross Listed': 'cross_listed',
    'Home Department': 'home_department',
    'PRIMARY\nClass Assoc Requisite, Enrollment Requirement Grp': 'primary_class_association_requisite_enrollment_requirement_group',
    'Additional\nClass Assoc Requisite, Enrollment Requirement Grp': 'additional_class_association_requisite_enrollment_requirement_group',
}, inplace=False)

# # Split the course title into long and short and save the long part
courses['course_title_long_short'] = courses['course_title_long_short'].apply(lambda x: x.split('\n')[0])
courses = courses.rename(columns={
    'course_title_long_short': 'course_title',
}, inplace=False)


def extract_prereqs(prereq_string):
    # Regular expression to find all occurrences of "SI" followed by three digits
    pattern = r'(\d{3})'  # The parentheses define a capturing group for the digits
    # Use findall to get all matching groups
    matches = re.findall(pattern, prereq_string)
    # Extract the digits from the matches
    # This will give you a list of strings like ['123', '456']
    matches = [match for match in matches if int(match) > 499]  # Remove 'SI ' from the matches
    matches = [match for match in matches if int(match) not in [501, 521, 601]]  # Remove courses that don't exist anymore
    # Convert the string digits to integers
    # int_matches = [int(match) for match in matches]
    return matches

# Apply the function to the 'prerequisites' column
courses['prerequisites'] = courses['prerequisites'].apply(lambda x: extract_prereqs(x) if isinstance(x, str) else None)

courses["catalog_number"] = courses["catalog_number"].astype("str")

# remove duplicate numbers in the prerequisites list
courses['prerequisites'] = courses['prerequisites'].apply(lambda x: list(set(x)) if isinstance(x, list) else x)


courses['term_offerings_2025_26'] = courses['term_offerings_2025_26'].str.split('\n')
courses['typical_term_offered'] = courses['typical_term_offered'].astype("str").apply(lambda x: x.split('\n')[0])

# # drop a column
courses = courses.drop(columns=['Term Offered in \n2023-24AY'])
courses = courses.drop(columns=['Term Offered in 2022-23AY'])
courses = courses.drop(columns=['Most Recent Syllabus\n'])

# remove all columns that arent catalog numbner, course title, preqrequisites, and term offerings, and desciption
courses = courses[['catalog_number', 'course_title', 'prerequisites', 'typical_term_offered', 'course_description', 'Credits']]
courses['prerequisites'] = courses['prerequisites'].apply(lambda x: json.dumps(x) if isinstance(x, list) else x)

# Function to extract prerequisites from a string




courses.sample(10)

,catalog_number,course_title,prerequisites,typical_term_offered,course_description,Credits
69,649,Information Visualization,"[""507"", ""544""]",Fall,Introduction to information visualization. Top...,3
2,505,Career and Internship Studio: Design Your Succ...,None,Fall (2nd half),"In this course, students will engage in a synt...",1
1,504,"Servers, The Shell, and Git",None,Fall,This course will introduce students to common ...,1.5
57,627,Managing Information Technology,None,Alternating Winter terms,Students will learn the roles and functions th...,3
41,599,SI Project Experience,[],nan,This project-based course requires students to...,3
71,652,Incentives and Strategic Behavior in Computati...,None,--,Modeling and analysis of strategic decision en...,3
91,688,Immersive Applied Projects in the Social Sector,None,Winter (Coursework completed over summer),"This course is an innovative, immersive experi...",6
44,602,Mathematical Foundations for Applied Data Science,"[""506"", ""504"", ""544""]",Winter,This course builds and strengthens the mathema...,3
60,632,Appraisal and Collection Development,"[""647"", ""580""]",Fall,Covers concepts and practices collection devel...,3
4,507,Intermediate Programming,"[""506"", ""504""]",Fall,The purpose of this course is to build upon th...,3


In [64]:
# write the csv to file
courses.to_csv('msi_courses_cleaned.csv', index=False)

## Load cleaned data

In [ ]:
import pandas as pd

cleaned_courses = pd.read_csv('msi_courses_cleaned.csv')

array([['SI', '500',
        'Problem-solving with People, Information, & Technology', ...,
        nan, '001507-MSI & MHI SI Pgms', nan],
       ['SI', '504', 'Servers, The Shell, and Git ', ..., nan,
        '001507-MSI & MHI SI Pgms',
        '10X sections: \n000032 - Graduate Standing'],
       ['SI', '505',
        'Career and Internship Studio: Design Your Success ', ..., nan,
        '001507-MSI & MHI SI Pgms', nan],
       ...,
       ['SI', '990', 'Diss-Precand', ..., nan, nan, nan],
       ['SI', '995', 'Diss-Cand', ..., nan, nan, nan],
       ['SI', '998', 'Curriculum Practical Project ', ..., nan, nan, nan]],
      shape=(119, 15), dtype=object)